# Regridding random forest predictions to 0.25° resolution

In [ ]:
# Libraries
import os
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'
dir05 = '../paper_ecosystem_carbon/output/05_rf_constrain/'

---

### Pre-processing

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

In [ ]:
# Export dataframes as nc-files for regridding
for ctype in ['cveg', 'csoil']:
    for etype in list_etype:
        # Get data
        ds = xr.open_zarr(f'{dir03}ds_rfpred_{ctype}_r001.zarr')

        # Replace nan
        ds = ds.where(ds != -32768)
        
        # Add lat and lon attributes (necessary for regridding)
        ds.lat.attrs = dict(standard_name = 'latitude',
                                    long_name = 'latitude',
                                    units = 'degrees_north',
                                    axis = 'Y')
        
        ds.lon.attrs = dict(standard_name = 'longitude',
                                    long_name = 'longitude',
                                    units = 'degrees_east',
                                    axis = 'X')

        # Export
        ds[f'{ctype}_{etype}'] \
            .to_netcdf(f'{dir05}ds_rfpred_{ctype}_{etype}_r001.nc', mode='w')

---

### Regridding

In [ ]:
%%bash 
cd ../paper_ecosystem_carbon/output/05_rf_constrain/

module load cdo

for i in \
    trop_ef trop_df temp_ebf temp_dbf temp_enf temp_dnf glob_shrub glob_grass \
    glob_tundr glob_mount glob_crop glob_past
do
    cdo -P 24 remapycon,../data/luh2_v2h/states.nc \
    ds_rfpred_cveg_${i}_r001.nc ds_rfpred_cveg_${i}_rcon_r025.nc

done

In [ ]:
%%bash 
cd ../paper_ecosystem_carbon/output/05_rf_constrain/

module load cdo

for i in \
    trop_ef trop_df temp_ebf temp_dbf temp_enf temp_dnf glob_shrub glob_grass \
    glob_tundr glob_mount glob_crop glob_past
do
    cdo -P 24 remapycon,../data/luh2_v2h/states.nc \
    ds_rfpred_csoil_${i}_r001.nc ds_rfpred_csoil_${i}_rcon_r025.nc

done

---

### Post-processing

In [ ]:
# Merge regridded rcon files to one dataset
for ctype in ['cveg', 'csoil']:
    xr.open_mfdataset(f'{dir05}ds_rfpred_{ctype}_*_rcon_r025.nc') \
        .to_zarr(f'{dir05}ds_rfpred_{ctype}_rcon_r025.zarr', mode='w')

In [ ]:
# Delete intermediate files
for ctype in ['cveg', 'csoil']:
    for etype in list_etype:
        os.remove(f'{dir05}ds_rfpred_{ctype}_{etype}_r001.nc')
        os.remove(f'{dir05}ds_rfpred_{ctype}_{etype}_rcon_r025.nc')